# 04 Size Stroke Augmentation

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb005_balanced_scale_stroke.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb005_balanced_scale_stroke.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb006_scale40_50_stroke.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb006_scale40_50_stroke.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb007_scale60_100_stroke.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb007_scale60_100_stroke.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb005_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb005_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb006_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb006_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb006_cross_compare.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb006_cross_compare.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb007_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb007_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb007_cross_compare.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb007_cross_compare.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb005_balanced_scale_stroke.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'e7001412011ed1ce928a9686d57a23afeb534769'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb006_scale40_50_stroke.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'bcd39ee27d88d6c92fdad89279e72dd640da6adb'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb007_scale60_100_stroke.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'a34634b06420fe5a62a5bf5623bd4c5bbe838c6a'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb005_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '23827f78c2dd3e7721ce5704f3170dcd05e5c89e'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb006_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'cc7e09587f2fceb3783e0dc121ca8322ac76e3ba'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb006_cross_compare.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '50ad41093ac9b202baa12db668f4e16a42b6a8f9'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb007_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '36027cf39af66bc7aea1bfafb50f7ce310e69704'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb007_cross_compare.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '7807333683e199a4194fcfab59588f4ab2cfc43f'}}

## exp_cmb005_balanced_scale_stroke

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb005_balanced_scale_stroke.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb005_balanced_scale_stroke.py)  
Git blob: `e7001412011ed1ce928a9686d57a23afeb534769`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-005. Match Normal's class-balanced Scale/Stroke, retain Combined writer CV.

Ten pre-registered low-ratio arms: com_only plus 10/20/30 percent Scale
crossed with 0/14/28 percent topology-preserving Stroke. Model hyperparameters fixed and equal across arms; 3 seeds
42/43/44; 4 held-out writers per arm. No Original/Professor Test or new 480.
No fitting, augmentation, or selection using held-out writers/OfficialVal.

CLI 인자 (정적 추출): `--arm`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb005_balanced_scale_stroke.py
#!/usr/bin/env python3
"""EXP-CMB-005. Match Normal's class-balanced Scale/Stroke, retain Combined writer CV.

Ten pre-registered low-ratio arms: com_only plus 10/20/30 percent Scale
crossed with 0/14/28 percent topology-preserving Stroke. Model hyperparameters fixed and equal across arms; 3 seeds
42/43/44; 4 held-out writers per arm. No Original/Professor Test or new 480.
No fitting, augmentation, or selection using held-out writers/OfficialVal.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,sys,time,warnings
from pathlib import Path
import joblib
import numpy as np
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,roc_auc_score,
                             precision_recall_fscore_support,confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
NORMAL=ROOT/"05_model_optimize"/"MLP"/"Normal"
sys.path.insert(0,str(CMB/"scripts"))
sys.path.insert(0,str(NORMAL/"scripts"))
from combined_mlp_writer_cv import (load_official,load_writers,PixelScaler255,
                                    WRITERS,LABELS,V3_SHA,HANDMADE_SHA)
from exp006_size_com_comparison import process,image_geometry
from exp015_scale_stroke_grid import nested_subset
from exp016_high_fraction_grid import scale_base,stroke_jitter
from exp014_balanced_augmentation import balanced_indices
from exp011_012_causal_probes import fixed_scale

SCALES=(.8,.9,1.,1.1,1.2)
PAIRS=(("0","D"),("D","0"),("8","B"),("B","8"),("5","F"),("F","5"),
       ("7","9"),("7","A"),("4","F"))
ARMS={"com_only":(0.,0.),
      "scale10_stroke00":(.1,0.),
      "scale10_stroke14":(.1,.14),
      "scale10_stroke28":(.1,.28),
      "scale20_stroke00":(.2,0.),
      "scale20_stroke14":(.2,.14),
      "scale20_stroke28":(.2,.28),
      "scale30_stroke00":(.3,0.),
      "scale30_stroke14":(.3,.14),
      "scale30_stroke28":(.3,.28)}
TARGETS=(16,18,20,22,24)
SEEDS=(42,43,44)
CLASSES=np.arange(16,dtype=int)

def savecsv(p,records):
    records=list(records)
    if not records:raise RuntimeError("missing rows "+str(p))
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        wr=csv.DictWriter(f,fieldnames=list(records[0]))
        wr.writeheader();wr.writerows(records)

def metrics(y,p,subset=None):
    labels=CLASSES if subset is None else np.asarray(subset)
    prec,rec,f1,sup=precision_recall_fscore_support(y,p,labels=labels,zero_division=0)
    return {"n":int(len(y)),"correct":int(np.sum(p==y)),
            "accuracy":float(np.mean(p==y)),
            "macro_f1":float(f1.mean()),
            "macro_precision":float(prec.mean()),
            "macro_recall":float(rec.mean())}

def confusion_pairs(y,p,writer,arm,seed,source):
    """All 240 off-diagonal directions, including zero-error pairs.

    Denominator n_actual corresponds to the *actual* (row) class.
    Both counts and conditional percentages are saved; total overall
    accuracy does not replace per-class and per-pair error analysis.
    """
    cm=confusion_matrix(y,p,labels=CLASSES)
    result=[]
    nall=int(len(y))
    nwrong=int(nall-np.trace(cm))
    for actual in CLASSES:
        nactual=int(cm[actual].sum())
        wrong=int(nactual-cm[actual,actual])
        for target in CLASSES:
            if actual==target:continue
            ct=int(cm[actual,target])
            result.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
                 "actual":LABELS[actual],"predicted":LABELS[target],
                 "n_actual":nactual,"n_actual_wrong":wrong,
                 "n_dataset":nall,"n_dataset_wrong":nwrong,
                 "errors":ct,"error_rate_within_actual_class":ct/nactual if nactual else 0,
                 "share_of_actual_class_errors":ct/wrong if wrong else 0,
                 "share_of_all_dataset_errors":ct/nwrong if nwrong else 0})
    assert len(result)==16*15
    assert sum(int(r["errors"]) for r in result)==nwrong
    return result

def total_errors_by_class(y,p,writer,arm,seed,source):
    """All 16 true classes: total incorrect / n actual class, plus group switch."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    out=[]
    for a in CLASSES:
        n=int(cm[a].sum());correct=int(cm[a,a]);wrong=n-correct
        family_swap=int(cm[a,10:].sum()) if a<10 else int(cm[a,:10].sum())
        out.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
           "actual":LABELS[a],"n":n,"correct":correct,"wrong":wrong,
           "class_error_rate":wrong/n if n else 0,
           "opposite_family_errors":family_swap,
           "opposite_family_error_rate":family_swap/n if n else 0})
    assert len(out)==16
    return out

def family_error_breakdown(y,p,writer,arm,seed,source):
    """Digits→letters and letters→digits alongside within-family mistakes."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    d2l=int(cm[:10,10:].sum());l2d=int(cm[10:,:10].sum())
    d2d=int(cm[:10,:10].sum()-np.trace(cm[:10,:10]))
    l2l=int(cm[10:,10:].sum()-np.trace(cm[10:,10:]))
    return [{"arm":arm,"seed":seed,"dataset":source,"writer":writer,
      "direction":key,"errors":val,"n_actual_family":n,
      "error_rate":val/n if n else 0}
      for key,val,n in (("digit_to_letter",d2l,int(cm[:10].sum())),
                        ("letter_to_digit",l2d,int(cm[10:].sum())),
                        ("digit_to_digit",d2d,int(cm[:10].sum())),
                        ("letter_to_letter",l2l,int(cm[10:].sum())))]

def geom_rows(name,x,y,arm,seed):
    geo=image_geometry(x)
    out=[]
    for nameg,mask in (("digits",y<10),("letters",y>=10)):
        z={"arm":arm,"seed":seed,"dataset":name,"group":nameg,
           "n":int(mask.sum())}
        for field in ("long_side","width","height","mass"):
            zz=np.asarray(geo[field])[mask]
            z[field+"_mean"]=float(zz.mean())
            z[field+"_std"]=float(zz.std())
        out.append(z)
    return out

def binary_size_probe(xtrain,ytrain,xval,yval):
    train=image_geometry(xtrain)["long_side"].reshape(-1,1)
    val=image_geometry(xval)["long_side"].reshape(-1,1)
    model=Pipeline([("std",StandardScaler()),
      ("logistic",LogisticRegression(class_weight="balanced",random_state=42,max_iter=1000))])
    model.fit(train,(ytrain>=10).astype(int))
    q=model.predict_proba(val)[:,1]
    return {"bal_acc":float(balanced_accuracy_score(yval>=10,q>=.5)),
            "auc":float(roc_auc_score(yval>=10,q))}

def prepareraw(official,writers,arm):
    ratio,stroke=ARMS[arm]
    raw=official["X_train"]
    yy=official["y_train"]
    # CoM-only is exactly the EXP-CMB-004 reference condition.
    main=process(raw,"com",20)
    changed={"method":arm,"requested_ratio":ratio,"actual_ratio":0.,
             "n_aug":0,"n_stroke_changed":0,"n_stroke_rejected":0,
             "official_train_rows":len(raw)}
    targets=[]
    if ratio:
        ids,target_map=nested_subset(yy,ratio)  # Normal EXP015 nested low-ratio group
        half,_=balanced_indices(yy,seed=20261116)
        augmented=scale_base(raw,ids,target_map)
        augmented,stroke_stats=stroke_jitter(augmented,stroke,ids,half)
        main[ids]=process(augmented,"com",20)
        changed.update({"n_aug":len(ids),"actual_ratio":len(ids)/len(yy),
             "n_stroke_changed":stroke_stats["modified"],
             "n_stroke_rejected":stroke_stats["rejected"]})
        for cl in CLASSES:
            for length in TARGETS:
                ct=int(np.sum((yy[ids]==cl)&(target_map[ids]==length)))
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":ct})
                assert ct>=1
        for cl in CLASSES:
            cc=[x["train_images_assigned"] for x in targets if x["class"]==LABELS[cl]]
            assert max(cc)-min(cc)<=1
        assert abs(len(ids)/len(yy)-ratio)<.002
    else:
        for cl in CLASSES:
            for length in TARGETS:
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":0})
    transformed_w={w:process(writers[w]["X"],"com",20) for w in WRITERS}
    assert main.shape==(76800,784) and len(transformed_w)==4
    return main,transformed_w,changed,targets

def make_scaled(official,writers):
    ref={}
    for scale in SCALES:
        # exact Normal EXP012 intervention; raw image BBox-centered affine, then CoM
        ref[("official",scale)]=process(fixed_scale(official["X_val"],scale),"com",20)
        for w in WRITERS:
            ref[(w,scale)]=process(fixed_scale(writers[w]["X"],scale),"com",20)
    return ref

def eval_scales(model,raw_map,y,dataset,writer,arm,seed):
    rows=[]; pair_rows=[]; focus=[]; class_errors=[]; family_errors=[]
    probs={}
    for sc in SCALES:
        xx=raw_map[(writer if dataset=="OOF" else "official",sc)]
        q=np.asarray(model.predict_proba(xx),dtype=np.float64)
        assert np.array_equal(model.named_steps["model"].classes_,CLASSES)
        p=q.argmax(axis=1).astype(int)
        letter=q[:,10:].sum(axis=1)
        probs[sc]=(p,letter)
    _,base_letter=probs[1.]
    base_family=base_letter>=.5
    for sc in SCALES:
        p,letter=probs[sc]
        for group,z,cls in (("all",np.ones(len(y),dtype=bool),CLASSES),
             ("digits",y<10,np.arange(10)),("letters",y>=10,np.arange(10,16))):
            m=metrics(y[z],p[z],cls)
            flip=(letter[z]>=.5)!=(base_family[z])
            rows.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
               "scale":sc,"group":group,**m,
               "family_flip":float(flip.mean()),
               "mean_abs_delta_p_letter":float(np.abs(letter[z]-base_letter[z]).mean()),
               "mean_p_letter":float(letter[z].mean())})
        for cls_i in CLASSES:
            z=y==cls_i
            pz=p[z];ll=letter[z];bb=base_letter[z]
            focus.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
                "scale":sc,"actual":LABELS[cls_i],"n":int(z.sum()),
                "correct":int(np.sum(pz==cls_i)),
                "accuracy":float(np.mean(pz==cls_i)),
                "family_flip":float(np.mean((ll>=.5)!=(bb>=.5))),
                "mean_abs_delta_p_letter":float(np.mean(np.abs(ll-bb))),
                "mean_p_letter":float(ll.mean())})
        sc_name=dataset+"_scale_"+str(sc)
        pair_rows+=confusion_pairs(y,p,writer,arm,seed,sc_name)
        class_errors+=total_errors_by_class(y,p,writer,arm,seed,sc_name)
        family_errors+=family_error_breakdown(y,p,writer,arm,seed,sc_name)
    return rows,pair_rows,focus,probs[1.][0],class_errors,family_errors

def output_plots(y,p,out,arm,seed):
    cm=confusion_matrix(y,p,labels=CLASSES)
    for k in ("counts","row_normalized"):
        v=cm if k=="counts" else 100*cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(10,8))
        im=ax.imshow(v,cmap="viridis",vmin=0,vmax=160 if k=="counts" else 100)
        fig.colorbar(im,ax=ax,fraction=.045,pad=.04)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),
               xticklabels=list(LABELS),yticklabels=list(LABELS),
               xlabel="Predicted",ylabel="True",
               title=f"EXP-CMB-005 {arm} seed={seed} Combined OOF {k}")
        fig.tight_layout()
        fig.savefig(out/f"OOF_confusion_{k}.png",dpi=145)
        plt.close(fig)

def run(arm,seed,out):
    if arm not in ARMS or seed not in SEEDS:raise ValueError((arm,seed))
    if out.exists(): raise FileExistsError(out)
    out.mkdir(parents=True)
    (out/"models").mkdir()
    t0=time.perf_counter()
    original=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
    handmade=ROOT/"02_data"/"Hand_made"
    off=load_official(original)
    writers=load_writers(handmade)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    assert len(off["X_train"])==76800 and len(off["X_val"])==19200
    tpre=time.perf_counter()
    xtrain_off,xwriters,info,targets=prepareraw(off,writers,arm)
    xval=process(off["X_val"],"com",20)
    scale_map=make_scaled(off,writers)
    preprocess_time=time.perf_counter()-tpre
    official_probe=binary_size_probe(xtrain_off,off["y_train"],xval,off["y_val"])
    savecsv(out/"target_assignments.csv",targets)
    features=geom_rows("official_train",xtrain_off,off["y_train"],arm,seed)
    features+=geom_rows("official_val",xval,off["y_val"],arm,seed)
    for w in WRITERS:features+=geom_rows("writer_"+str(w),xwriters[w],writers[w]["y"],arm,seed)
    savecsv(out/"geometry_distributions.csv",features)
    foldrows=[];pairs=[];class_errors=[];family_errors=[];scale_rows=[];class_rows=[];scale_class_rows=[];pred_rows=[];loss=[];fit=[]
    pooled=[]
    for held in WRITERS:
        trwriters=[w for w in WRITERS if w!=held]
        X=np.concatenate([xtrain_off]+[xwriters[w] for w in trwriters])
        y=np.concatenate([off["y_train"]]+[writers[w]["y"] for w in trwriters])
        assert X.shape==(78720,784) and len(y)==78720
        mlp=Pipeline([("scale_0_1",PixelScaler255()),
                      ("model",MLPClassifier(random_state=seed))])
        started=time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mlp.fit(X,y)
        sec=time.perf_counter()-started
        mlpnet=mlp.named_steps["model"]
        assert mlpnet.n_features_in_==784
        for it,lossval in enumerate(mlpnet.loss_curve_):
            loss.append({"arm":arm,"seed":seed,"heldout":held,
                        "iteration":it+1,"loss":float(lossval)})
        fit.append({"arm":arm,"seed":seed,"heldout":held,
                    "train_n":len(y),"heldout_n":640,
                    "fit_seconds":sec,"iterations":mlpnet.n_iter_,
                    "last_loss":float(mlpnet.loss_),
                    "warnings":" | ".join(str(w.message) for w in caught)})
        fout=out/"models"/f"heldout_{held}.joblib"
        joblib.dump(mlp,fout,compress=3)
        handwritten_y=writers[held]["y"]
        outcome=mlp.predict(xwriters[held]).astype(int)
        pp=mlp.predict(xval).astype(int)
        for ds,ytrue,pred in (("OOF",handwritten_y,outcome),
                              ("OfficialVal",off["y_val"],pp)):
            maskset=(("all",np.ones(len(ytrue),bool),CLASSES),
                     ("digits",ytrue<10,np.arange(10)),
                     ("letters",ytrue>=10,np.arange(10,16)))
            for g,z,classes in maskset:
                foldrows.append({"arm":arm,"seed":seed,"heldout":held,"dataset":ds,
                      "group":g,**metrics(ytrue[z],pred[z],classes)})
        prec,rec,f1,_=precision_recall_fscore_support(handwritten_y,outcome,
              labels=CLASSES,zero_division=0)
        for ci in CLASSES:
            z=handwritten_y==ci
            class_rows.append({"arm":arm,"seed":seed,"writer":held,
               "class":LABELS[ci],"n":int(z.sum()),"correct":int(np.sum(outcome[z]==ci)),
               "precision":float(prec[ci]),"recall":float(rec[ci]),"f1":float(f1[ci])})
        for i in range(640):
            pred_rows.append({"arm":arm,"seed":seed,"writer":held,"index":i,
                "actual":LABELS[int(handwritten_y[i])],
                "pred":LABELS[int(outcome[i])]})
        pooled.append(outcome)
        pairs+=confusion_pairs(handwritten_y,outcome,held,arm,seed,"OOF_original")
        pairs+=confusion_pairs(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        class_errors+=total_errors_by_class(handwritten_y,outcome,held,arm,seed,"OOF_original")
        class_errors+=total_errors_by_class(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        family_errors+=family_error_breakdown(handwritten_y,outcome,held,arm,seed,"OOF_original")
        family_errors+=family_error_breakdown(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        for ds,ytrue,wid in (("OOF",handwritten_y,held),
                             ("OfficialVal",off["y_val"],held)):
            z,p2,c,base,ce,fe=eval_scales(mlp,scale_map,ytrue,ds,wid,arm,seed)
            scale_rows+=z;pairs+=p2;scale_class_rows+=c
            class_errors+=ce;family_errors+=fe
            if ds=="OOF":assert np.array_equal(base,outcome)
            else:assert np.array_equal(base,pp)
        print("FOLD_COMPLETED",arm,seed,held,
              "OOF",round(np.mean(outcome==handwritten_y),6),
              "VAL",round(np.mean(pp==off["y_val"]),6),
              "fit_s",round(sec,2),flush=True)
        del X,y,mlp
    hand_y=np.concatenate([writers[w]["y"] for w in WRITERS])
    pooled_p=np.concatenate(pooled)
    assert len(hand_y)==2560
    output_plots(hand_y,pooled_p,out,arm,seed)
    savecsv(out/"fold_metrics.csv",foldrows)
    savecsv(out/"oof_per_class.csv",class_rows)
    savecsv(out/"per_class_and_scale.csv",scale_class_rows)
    savecsv(out/"pair_confusions.csv",pairs)
    savecsv(out/"per_class_error_rates.csv",class_errors)
    savecsv(out/"family_confusion_rates.csv",family_errors)
    savecsv(out/"scale_sensitivity.csv",scale_rows)
    savecsv(out/"oof_predictions.csv",pred_rows)
    savecsv(out/"training_loss.csv",loss)
    savecsv(out/"training_time.csv",fit)
    overall=metrics(hand_y,pooled_p)
    meta={"experiment":"EXP-CMB-005","status":"arm completed, aggregation pending",
          "arm":arm,"seed":seed,"handmade_cv":"Leave-One-Writer-Out 4-Fold",
          "official_n":76800,"official_val_n":19200,"training_handmade_each_fold":1920,
          "oof_n":2560,"scale_interventions":SCALES,"class_balanced_targets":TARGETS,
          "official_train_only_augmentation":True,
          "handmade_train_preprocess":"CoM-only",
          "heldout_and_official_val_preprocess":"CoM-only",
          "augmentation_seed":20261116,"augmentation":info,
          "preregistration":"low ratio 0,10,20,30 x stroke 0,14,28 (no stroke for 0)",
          "no_mlp_hyperparameter_search":True,
          "official_sha256":off["sha256"],
          "handmade_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
          "official_train_size_binary_probe":official_probe,
          "oof_metrics":overall,"fit_total_seconds":sum(r["fit_seconds"] for r in fit),
          "preprocessing_seconds":preprocess_time,
          "wall_seconds":time.perf_counter()-t0,
          "original_test_used":False,"professor_test_used":False,
          "new_handmade_480_used":False,
          "versions":{"numpy":np.__version__,"scikit_learn":sklearn.__version__},
          "model_sha256":{str(w):hashlib.sha256((out/"models"/f"heldout_{w}.joblib").read_bytes()).hexdigest()
                          for w in WRITERS}}
    (out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("COMPLETED",arm,seed,json.dumps(overall),flush=True)

if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=tuple(ARMS),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    ar=ap.parse_args()
    run(ar.arm,ar.seed,ar.out)


## exp_cmb006_scale40_50_stroke

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb006_scale40_50_stroke.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb006_scale40_50_stroke.py)  
Git blob: `bcd39ee27d88d6c92fdad89279e72dd640da6adb`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-006: fixed-MLP Size augmentation 40/50% x Stroke 0/14/28%.

Reuse the already validated EXP-CMB-005 implementation for all preprocessing,
label splits, raw-dataset SHA checks, OOF 16-class/240-pair diagnostics,
5-scale probes, and model weights. Changing only the 6-arm design ensures that
CMB005 historical code stays completely unchanged.

CLI 인자 (정적 추출): `--arm`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb006_scale40_50_stroke.py
#!/usr/bin/env python3
"""EXP-CMB-006: fixed-MLP Size augmentation 40/50% x Stroke 0/14/28%.

Reuse the already validated EXP-CMB-005 implementation for all preprocessing,
label splits, raw-dataset SHA checks, OOF 16-class/240-pair diagnostics,
5-scale probes, and model weights. Changing only the 6-arm design ensures that
CMB005 historical code stays completely unchanged.
"""
from __future__ import annotations
import argparse
import json
from pathlib import Path
import exp_cmb005_balanced_scale_stroke as baseline

ARMS={
    "scale40_stroke00":(.4,0.),
    "scale40_stroke14":(.4,.14),
    "scale40_stroke28":(.4,.28),
    "scale50_stroke00":(.5,0.),
    "scale50_stroke14":(.5,.14),
    "scale50_stroke28":(.5,.28),
}
SEEDS=(42,43,44)

def main():
    ap=argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--arm",choices=tuple(ARMS),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    args=ap.parse_args()

    # EXP-CMB-005 run/prepareraw consult this module-level mapping dynamically.
    baseline.ARMS.clear()
    baseline.ARMS.update(ARMS)
    baseline.run(args.arm,args.seed,args.out)

    mf=args.out/"manifest.json"
    info=json.loads(mf.read_text(encoding="utf-8"))
    assert info["experiment"]=="EXP-CMB-005"
    assert info["arm"]==args.arm and info["seed"]==args.seed
    assert len(info["model_sha256"])==4
    assert info["official_n"]==76800
    assert info["official_val_n"]==19200
    assert info["training_handmade_each_fold"]==1920
    assert info["oof_n"]==2560
    assert not info["original_test_used"] and not info["professor_test_used"]
    assert not info["new_handmade_480_used"]
    assert info["no_mlp_hyperparameter_search"]
    requested=ARMS[args.arm][0]
    assert abs(info["augmentation"]["actual_ratio"]-requested)<.002

    info.update({
        "experiment":"EXP-CMB-006",
        "reference_experiment":"EXP-CMB-005",
        "reference_script":"exp_cmb005_balanced_scale_stroke.py",
        "pre_registered_arm_set":ARMS,
        "fixed_model_hyperparameters":True,
        "independent_new_models":4,
        "total_planned_fits":72,
        "pre_registered_size_rates":[.4,.5],
        "pre_registered_stroke_strengths":[0.,.14,.28],
        "controls":"Compare paired seeds/writers/images with previously frozen CMB005",
        "original_test_used":False,
        "professor_test_used":False,
        "model_selection":"none; metrics explanatory only"
    })
    mf.write_text(json.dumps(info,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("EXP_CMB_006_COMPLETED",args.arm,args.seed,info["oof_metrics"],
          flush=True)

if __name__=="__main__":
    main()


## exp_cmb007_scale60_100_stroke

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb007_scale60_100_stroke.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb007_scale60_100_stroke.py)  
Git blob: `a34634b06420fe5a62a5bf5623bd4c5bbe838c6a`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-007: fixed-MLP Size augmentation 60/70/80/90/100% x Stroke 0/14/28%.

Reuse the already validated EXP-CMB-005 implementation for all preprocessing,
label splits, raw-dataset SHA checks, OOF 16-class/240-pair diagnostics,
5-scale probes, and model weights. Changing only the 6-arm design ensures that
CMB005 historical code stays completely unchanged.

CLI 인자 (정적 추출): `--arm`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb007_scale60_100_stroke.py
#!/usr/bin/env python3
"""EXP-CMB-007: fixed-MLP Size augmentation 60/70/80/90/100% x Stroke 0/14/28%.

Reuse the already validated EXP-CMB-005 implementation for all preprocessing,
label splits, raw-dataset SHA checks, OOF 16-class/240-pair diagnostics,
5-scale probes, and model weights. Changing only the 6-arm design ensures that
CMB005 historical code stays completely unchanged.
"""
from __future__ import annotations
import argparse
import json
from pathlib import Path
import exp_cmb005_balanced_scale_stroke as baseline

from exp016_high_fraction_grid import nested_subset as high_fraction_nested_subset

ARMS={f"scale{pct:02d}_stroke{stroke:02d}":(pct/100,stroke/100)
      for pct in (60,70,80,90,100) for stroke in (0,14,28)}
SEEDS=(42,43,44)

def main():
    ap=argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--arm",choices=tuple(ARMS),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    args=ap.parse_args()

    # EXP-CMB-005 run/prepareraw consult this module-level mapping dynamically.
    # Use Normal EXP016 high fraction selections, not Normal EXP015 <=50%.
    # This is a controlled extension preserving all historical 50% source IDs.
    baseline.nested_subset=high_fraction_nested_subset
    baseline.ARMS.clear()
    baseline.ARMS.update(ARMS)
    baseline.run(args.arm,args.seed,args.out)

    mf=args.out/"manifest.json"
    info=json.loads(mf.read_text(encoding="utf-8"))
    assert info["experiment"]=="EXP-CMB-005"
    assert info["arm"]==args.arm and info["seed"]==args.seed
    assert len(info["model_sha256"])==4
    assert info["official_n"]==76800
    assert info["official_val_n"]==19200
    assert info["training_handmade_each_fold"]==1920
    assert info["oof_n"]==2560
    assert not info["original_test_used"] and not info["professor_test_used"]
    assert not info["new_handmade_480_used"]
    assert info["no_mlp_hyperparameter_search"]
    requested=ARMS[args.arm][0]
    assert abs(info["augmentation"]["actual_ratio"]-requested)<.002

    info.update({
        "experiment":"EXP-CMB-007",
        "reference_experiment":"EXP-CMB-005",
        "reference_script":"exp_cmb005_balanced_scale_stroke.py",
        "pre_registered_arm_set":ARMS,
        "fixed_model_hyperparameters":True,
        "independent_new_models":4,
        "total_planned_fits":180,
        "pre_registered_size_rates":[.6,.7,.8,.9,1.],
        "training_source_selection":"EXP016 high_fraction_grid nested_subset",
        "historical_first_50_percent_ids_retained":True,
        "pre_registered_stroke_strengths":[0.,.14,.28],
        "controls":"Compare paired seeds/writers/images with previously frozen CMB005",
        "original_test_used":False,
        "professor_test_used":False,
        "model_selection":"none; metrics explanatory only"
    })
    mf.write_text(json.dumps(info,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("EXP_CMB_007_COMPLETED",args.arm,args.seed,info["oof_metrics"],
          flush=True)

if __name__=="__main__":
    main()


## exp_cmb005_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb005_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb005_aggregate.py)  
Git blob: `23827f78c2dd3e7721ce5704f3170dcd05e5c89e`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Aggregate EXP-CMB-005 *completed* low-ratio variants. No model training.

Inputs are GitHub Actions artifacts with per-arm/per-seed run manifest and
per-fold predictions. Stage 1 requires 0 and 10% arms (4x3 jobs). Stage 2
requires 20/30% arms (6x3 jobs). The final combination can aggregate both.

CLI 인자 (정적 추출): `--incoming`, `--out`, `--stage`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb005_aggregate.py
#!/usr/bin/env python3
"""Aggregate EXP-CMB-005 *completed* low-ratio variants. No model training.

Inputs are GitHub Actions artifacts with per-arm/per-seed run manifest and
per-fold predictions. Stage 1 requires 0 and 10% arms (4x3 jobs). Stage 2
requires 20/30% arms (6x3 jobs). The final combination can aggregate both.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
ARMS={"com_only":(0.,0.),**{f"scale{pct:02d}_stroke{strength:02d}":(pct/100,strength/100)
        for pct in (10,20,30) for strength in (0,14,28)}}
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
LABELS="0123456789ABCDEF"
def loadcsv(p):
    with p.open(encoding="utf-8-sig",newline="") as f:return list(csv.DictReader(f))
def writecsv(p,rows):
    rows=list(rows);assert rows,p
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=rows[0].keys());w.writeheader();w.writerows(rows)
def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--incoming",type=Path,required=True)
    parser.add_argument("--out",type=Path,required=True)
    parser.add_argument("--stage",choices=("low10","low20to30","full"),required=True)
    args=parser.parse_args()
    expected={
      "low10":["com_only"]+[f"scale10_stroke{k:02d}" for k in (0,14,28)],
      "low20to30":[f"scale{p:02d}_stroke{k:02d}" for p in (20,30) for k in (0,14,28)],
      "full":list(ARMS)
    }[args.stage]
    out=args.out;out.mkdir(parents=True,exist_ok=True)
    original=out/"raw"
    allsummary=[];fold=[];pairs=[];class_error_rows=[];family_error_rows=[];responses=[];classes=[];geometry=[];perimage=[]
    for arm in expected:
        for seed in SEEDS:
            folder=args.incoming/f"cmb005-{arm}-s{seed}"
            manifest=json.loads((folder/"manifest.json").read_text(encoding="utf-8"))
            assert manifest["arm"]==arm and int(manifest["seed"])==seed
            assert not manifest["original_test_used"] and not manifest["professor_test_used"]
            assert not manifest["new_handmade_480_used"]
            assert manifest["no_mlp_hyperparameter_search"] if "no_mlp_hyperparameter_search" in manifest else True
            target=original/arm/f"seed_{seed}"
            target.mkdir(parents=True,exist_ok=True)
            for item in folder.iterdir():
                if item.is_file():
                    shutil.copyfile(item,target/item.name)
            pred=loadcsv(folder/"oof_predictions.csv")
            assert len(pred)==2560
            pp={int(x["writer"]):[] for x in pred}
            for x in pred:
                pp[int(x["writer"])].append(x)
            assert all(len(v)==640 for v in pp.values())
            yy=np.array([LABELS.index(x["actual"]) for x in pred])
            predicted=np.array([LABELS.index(x["pred"]) for x in pred])
            pr,re,f1,_=precision_recall_fscore_support(yy,predicted,
                labels=np.arange(16),zero_division=0)
            summary={"arm":arm,"seed":seed,
                "ratio":ARMS[arm][0],"stroke":ARMS[arm][1],
                "n":len(yy),"correct":int(np.sum(yy==predicted)),
                "accuracy":float(np.mean(yy==predicted)),
                "macro_f1":float(f1.mean()),
                "digits_acc":float(np.mean(yy[yy<10]==predicted[yy<10])),
                "letters_acc":float(np.mean(yy[yy>=10]==predicted[yy>=10]))}
            allsummary.append(summary)
            for f in loadcsv(folder/"fold_metrics.csv"):fold.append(f)
            for f in loadcsv(folder/"pair_confusions.csv"):pairs.append(f)
            for f in loadcsv(folder/"per_class_error_rates.csv"):class_error_rows.append(f)
            for f in loadcsv(folder/"family_confusion_rates.csv"):family_error_rows.append(f)
            for f in loadcsv(folder/"scale_sensitivity.csv"):responses.append(f)
            for f in loadcsv(folder/"per_class_and_scale.csv"):classes.append(f)
            for f in loadcsv(folder/"geometry_distributions.csv"):geometry.append(f)
            for x in pred:perimage.append(x)
            assert len(loadcsv(folder/"training_time.csv"))==4
            assert len(loadcsv(folder/"scale_sensitivity.csv"))==4*2*5*3
    assert len(allsummary)==len(expected)*3
    writecsv(out/"per_seed_overall.csv",allsummary)
    writecsv(out/"per_fold_group_metrics.csv",fold)
    writecsv(out/"pairwise_confusion_directions.csv",pairs)
    writecsv(out/"all_class_error_rates_raw.csv",class_error_rows)
    writecsv(out/"family_error_rates_raw.csv",family_error_rows)
    writecsv(out/"five_scale_group_responses.csv",responses)
    writecsv(out/"five_scale_16class_responses.csv",classes)
    writecsv(out/"trained_geometry.csv",geometry)
    writecsv(out/"oof_raw_predictions.csv",perimage)
    stats=[]
    for arm in expected:
        v=[x for x in allsummary if x["arm"]==arm]
        total=np.mean([x["accuracy"] for x in v])
        val=[x for x in fold if x["arm"]==arm and x["dataset"]=="OfficialVal"
            and x["group"]=="all"]
        normal=[x for x in responses if x["arm"]==arm and x["dataset"]=="OOF"
            and x["group"] in ("digits","letters")]
        risk=[]
        for group,scale in (("digits",1.2),("letters",.8)):
            current=[float(x["family_flip"]) for x in normal
                     if x["group"]==group and abs(float(x["scale"])-scale)<1e-6]
            assert len(current)==len(SEEDS)*len(WRITERS)
            risk.append(float(np.mean(current)))
        fcs=[x for x in classes if x["arm"]==arm and x["dataset"]=="OOF"
             and x["actual"] in ("B","8","D") and float(x["scale"]) in (.8,1.2)]
        worst=max(float(x["family_flip"]) for x in fcs) # single writer/seed, disclose
        fco=[x for x in classes if x["arm"]==arm and x["dataset"]=="OfficialVal"
             and x["actual"] in ("B","8","D") and float(x["scale"]) in (.8,1.2)]
        ov_worst=max(float(x["family_flip"]) for x in fco)
        stats.append({"arm":arm,"size_ratio":ARMS[arm][0],
            "stroke_strength":ARMS[arm][1],
            "mean_accuracy":total,
            "std_accuracy_seed":float(np.std([x["accuracy"] for x in v])),
            "mean_digits_acc":float(np.mean([x["digits_acc"] for x in v])),
            "mean_letters_acc":float(np.mean([x["letters_acc"] for x in v])),
            "official_val_accuracy":float(np.mean([float(x["accuracy"]) for x in val])),
            "digits_1p2_family_flip":risk[0],
            "letters_0p8_family_flip":risk[1],
            "max_writer_seed_B8D_family_flip":worst,
            "max_official_fold_seed_B8D_family_flip":ov_worst})
    writecsv(out/"comparison.csv",stats)
    # All 240 directional errors: aggregate across 4 held-out writers,
    # then average across 3 seeds. Percentages always use the source class
    # denominator (160 Hand-made per class per seed).
    baselinepath=Path(__file__).resolve().parents[4]/"05_model_optimize"/"MLP"/"Combined"/"results"/"EXP_CMB_001"/"OOF_predictions.csv"
    baseline=loadcsv(baselinepath)
    assert len(baseline)==2560
    baseline_cm=np.zeros((16,16),dtype=int)
    for r in baseline:
        aa=LABELS.index(r["true_label"]);bb=LABELS.index(r["combined_pred"])
        baseline_cm[aa,bb]+=1
    assert (np.trace(baseline_cm),baseline_cm.sum())==(2080,2560)
    direction_comparisons=[]
    class_comparisons=[]
    for arm in expected:
        baseline_wrong={LABELS[i]:int(baseline_cm[i].sum()-baseline_cm[i,i]) for i in range(16)}
        arr=[r for r in pairs if r["arm"]==arm and r["dataset"]=="OOF_original"]
        rr=[r for r in class_error_rows if r["arm"]==arm and r["dataset"]=="OOF_original"]
        assert len(arr)==3*4*16*15,(arm,len(arr))
        assert len(rr)==3*4*16
        for i,letter in enumerate(LABELS):
            cr=[r for r in rr if r["actual"]==letter]
            assert len(cr)==12
            current=float(np.mean([float(r["class_error_rate"]) for r in cr]))
            base=float(baseline_wrong[letter]/160)
            class_comparisons.append({"arm":arm,"actual":letter,"n_actual_per_seed":160,
               "baseline_wrong":baseline_wrong[letter],
               "baseline_error_rate":base,"mean_wrong":float(np.mean(
                    [sum(int(z["wrong"]) for z in cr if int(z["seed"])==seed) for seed in SEEDS])),
               "mean_class_error_rate":current,"delta_error_rate_pp":100*(current-base)})
            for j,target in enumerate(LABELS):
                if i==j:continue
                group=[r for r in arr if r["actual"]==letter and r["predicted"]==target]
                assert len(group)==12,(arm,letter,target,len(group))
                seedtot=[sum(int(r["errors"]) for r in group if int(r["seed"])==seed) for seed in SEEDS]
                baseline_count=int(baseline_cm[i,j])
                avg=float(np.mean(seedtot))
                direction_comparisons.append({"arm":arm,"actual":letter,"predicted":target,
                    "n_actual_per_seed":160,"baseline_errors":baseline_count,
                    "baseline_rate":baseline_count/160,
                    "mean_errors":avg,"mean_error_rate":avg/160,
                    "delta_errors_vs_baseline":avg-baseline_count,
                    "delta_error_rate_pp":100*(avg-baseline_count)/160})
    writecsv(out/"all_240_direction_changes.csv",direction_comparisons)
    writecsv(out/"all_16_class_error_changes.csv",class_comparisons)
    # Every direction, even absent, remains in the CSV; top20 for quick review.
    top20=sorted(direction_comparisons,key=lambda r:(-r["mean_errors"],r["arm"],r["actual"],r["predicted"]))[:20]
    writecsv(out/"top_20_error_directions_across_conditions.csv",top20)
    family_summary=[]
    for arm in expected:
        for fam in ("digit_to_letter","letter_to_digit","digit_to_digit","letter_to_letter"):
            x=[r for r in family_error_rows if r["arm"]==arm and r["dataset"]=="OOF_original"
               and r["direction"]==fam]
            assert len(x)==12
            family_summary.append({"arm":arm,"direction":fam,
              "mean_error_count":float(np.mean([sum(int(z["errors"]) for z in x
                                            if int(z["seed"])==seed) for seed in SEEDS])),
              "mean_error_rate_by_actual_family":float(np.mean([float(z["error_rate"]) for z in x]))})
    writecsv(out/"family_error_direction_comparison.csv",family_summary)

    # Paired effects: retain same seed & writer & image index; 10% stage includes CoM.
    corrections=[]
    for i in range(len(expected)):
        for j in range(i+1,len(expected)):
            a=expected[i];b=expected[j]
            for seed in SEEDS:
                def get(am):
                    return {(int(x["writer"]),int(x["index"])):x
                         for x in perimage if x["arm"]==am and int(x["seed"])==seed}
                pa,pb=get(a),get(b)
                assert len(pa)==len(pb)==2560 and pa.keys()==pb.keys()
                old=np.array([pa[k]["pred"]==pa[k]["actual"] for k in sorted(pa)])
                new=np.array([pb[k]["pred"]==pb[k]["actual"] for k in sorted(pb)])
                corrections.append({"a":a,"b":b,"seed":seed,
                    "corrected":int(np.sum(~old&new)),
                    "induced":int(np.sum(old&~new)),
                    "net":int(new.sum()-old.sum())})
    writecsv(out/"paired_corrected_induced.csv",corrections)
    lines=[f"# EXP-CMB-005 — {args.stage} 실측 집계","",
      "**전처리 효과 분석용. MLP HP는 모든 조건에 고정. 특정 Accuracy/F1 숫자를 넘기면 합격시키는 연구가 아니다.**",
      "", "| 조건 | OOF Accuracy | 숫자 | 문자 | 공식 Val | 숫자×1.2 Family flip | 문자×0.8 Family flip |",
      "|---|---:|---:|---:|---:|---:|---:|"]
    for a in stats:
        lines.append("| {} | {:.3%} | {:.3%} | {:.3%} | {:.3%} | {:.2%} | {:.2%} |".format(
          a["arm"],a["mean_accuracy"],a["mean_digits_acc"],a["mean_letters_acc"],
          a["official_val_accuracy"],a["digits_1p2_family_flip"],
          a["letters_0p8_family_flip"]))
    lines+=["","## 16개 전체 클래스의 오분류율 (OOF, 각 실제 클래스 160장/Seed)",
      "","| 조건 | 실제 클래스 | Baseline 오분류율 | 새 모델 오분류율 | 변화 (pp) |",
      "|---|---|---:|---:|---:|"]
    for r in class_comparisons:
        lines.append("| {} | {} | {:.2%} | {:.2%} | {:+.2f} |".format(
            r["arm"],r["actual"],r["baseline_error_rate"],
            r["mean_class_error_rate"],r["delta_error_rate_pp"]))
    lines+=["","## 그 밖의 주요 오류 방향 (기존 Baseline에서 0↔D/8↔B 제외 상위 15개)",
      "","| 실제→예측 | 기존 건수/160 | 기존 방향 오류율 |"]
    lines+=["|---|---:|---:|"]
    other=[]
    for i,a in enumerate(LABELS):
        for j,b in enumerate(LABELS):
            if i==j or (a,b) in (("0","D"),("D","0"),("8","B"),("B","8")):
                continue
            other.append((int(baseline_cm[i,j]),a,b))
    for count,a,b in sorted(other,reverse=True)[:15]:
        lines.append("| {}→{} | {} | {:.2%} |".format(a,b,count,count/160))
    lines+=["","**모든 방향(16×15=240개)과 전처리 전후 변화는 [all_240_direction_changes.csv](all_240_direction_changes.csv).**",
      "**전체 클래스별 오류율과 0→D/8→B 외 새롭게 늘어난 혼동은 [all_16_class_error_changes.csv](all_16_class_error_changes.csv), [family_error_direction_comparison.csv](family_error_direction_comparison.csv)에도 기록.**",
      ""]
    lines+=["","**주의:** 표의 숫자/문자 Family flip은 이미지는 같은데 문자계열 P(letter)≥0.5 여부가 바뀐 비율. Accuracy와 다르다. 개별 클래스/작성자에서 반대 방향의 오류와 공식 Val 하락이 존재할 수 있다.",
      "",
      "관련 원본: per_seed_overall.csv, per_fold_group_metrics.csv, five_scale_16class_responses.csv, trained_geometry.csv, pairwise_confusion_directions.csv, paired_corrected_induced.csv.",
      "",
      "사전등록된 비교 질문에 대한 기전 해석/반례/연구 목적 달성 판정은 별도 최종 연구일지에서 한다. 원본 Test와 교수님 Test는 접근하지 않았다.", ""]
    (out/"SUMMARY.md").write_text("\n".join(lines),encoding="utf-8")
    fig,ax=plt.subplots(figsize=(10,5.2))
    xs=np.arange(len(stats))
    ax.plot(xs,[100*a["mean_accuracy"] for a in stats],marker="o",label="Held-out writer OOF")
    ax.plot(xs,[100*a["official_val_accuracy"] for a in stats],marker="s",label="Official Development Val")
    ax.set(xticks=xs,xticklabels=[a["arm"] for a in stats],
       ylabel="Accuracy (%)",title=f"EXP-CMB-005 {args.stage} | Fixed MLP HP")
    plt.setp(ax.get_xticklabels(),rotation=30,ha="right")
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(out/"comparison.png",dpi=170);plt.close(fig)
    (out/"manifest.json").write_text(json.dumps({"experiment":"EXP-CMB-005",
      "stage":args.stage,"arms":expected,"seeds":SEEDS,"folds":WRITERS,
      "expected_fits":len(expected)*len(SEEDS)*len(WRITERS),
      "no_hp_search":True,"no_test_used":True,
      "research_gate":"explanation not absolute accuracy",
      "status":"numeric comparison complete, explanatory decision pending"},
      ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)
if __name__=="__main__":
    main()


## exp_cmb006_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb006_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb006_aggregate.py)  
Git blob: `cc7e09587f2fceb3783e0dc121ca8322ac76e3ba`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Aggregate EXP-CMB-006 *completed* low-ratio variants. No model training.

Inputs are GitHub Actions artifacts with per-arm/per-seed run manifest and
per-fold predictions. Stage 1 requires 0 and 10% arms (4x3 jobs). Stage 2
requires 20/30% arms (6x3 jobs). The final combination can aggregate both.

CLI 인자 (정적 추출): `--incoming`, `--out`, `--stage`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb006_aggregate.py
#!/usr/bin/env python3
"""Aggregate EXP-CMB-006 *completed* low-ratio variants. No model training.

Inputs are GitHub Actions artifacts with per-arm/per-seed run manifest and
per-fold predictions. Stage 1 requires 0 and 10% arms (4x3 jobs). Stage 2
requires 20/30% arms (6x3 jobs). The final combination can aggregate both.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
ARMS={f"scale{pct:02d}_stroke{strength:02d}":(pct/100,strength/100)
      for pct in (40,50) for strength in (0,14,28)}
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
LABELS="0123456789ABCDEF"
def loadcsv(p):
    with p.open(encoding="utf-8-sig",newline="") as f:return list(csv.DictReader(f))
def writecsv(p,rows):
    rows=list(rows);assert rows,p
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=rows[0].keys());w.writeheader();w.writerows(rows)
def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--incoming",type=Path,required=True)
    parser.add_argument("--out",type=Path,required=True)
    parser.add_argument("--stage",choices=("extended",),required=True)
    args=parser.parse_args()
    expected=list(ARMS)
    out=args.out;out.mkdir(parents=True,exist_ok=True)
    original=out/"raw"
    allsummary=[];fold=[];pairs=[];class_error_rows=[];family_error_rows=[];responses=[];classes=[];geometry=[];perimage=[]
    for arm in expected:
        for seed in SEEDS:
            folder=args.incoming/f"cmb006-{arm}-s{seed}"
            manifest=json.loads((folder/"manifest.json").read_text(encoding="utf-8"))
            assert manifest["arm"]==arm and int(manifest["seed"])==seed
            assert manifest["experiment"]=="EXP-CMB-006" and manifest["fixed_model_hyperparameters"]
            assert not manifest["original_test_used"] and not manifest["professor_test_used"]
            assert not manifest["new_handmade_480_used"]
            assert manifest["no_mlp_hyperparameter_search"] if "no_mlp_hyperparameter_search" in manifest else True
            target=original/arm/f"seed_{seed}"
            target.mkdir(parents=True,exist_ok=True)
            for item in folder.iterdir():
                if item.is_file():
                    shutil.copyfile(item,target/item.name)
            pred=loadcsv(folder/"oof_predictions.csv")
            assert len(pred)==2560
            pp={int(x["writer"]):[] for x in pred}
            for x in pred:
                pp[int(x["writer"])].append(x)
            assert all(len(v)==640 for v in pp.values())
            yy=np.array([LABELS.index(x["actual"]) for x in pred])
            predicted=np.array([LABELS.index(x["pred"]) for x in pred])
            pr,re,f1,_=precision_recall_fscore_support(yy,predicted,
                labels=np.arange(16),zero_division=0)
            summary={"arm":arm,"seed":seed,
                "ratio":ARMS[arm][0],"stroke":ARMS[arm][1],
                "n":len(yy),"correct":int(np.sum(yy==predicted)),
                "accuracy":float(np.mean(yy==predicted)),
                "macro_f1":float(f1.mean()),
                "digits_acc":float(np.mean(yy[yy<10]==predicted[yy<10])),
                "letters_acc":float(np.mean(yy[yy>=10]==predicted[yy>=10]))}
            allsummary.append(summary)
            for f in loadcsv(folder/"fold_metrics.csv"):fold.append(f)
            for f in loadcsv(folder/"pair_confusions.csv"):pairs.append(f)
            for f in loadcsv(folder/"per_class_error_rates.csv"):class_error_rows.append(f)
            for f in loadcsv(folder/"family_confusion_rates.csv"):family_error_rows.append(f)
            for f in loadcsv(folder/"scale_sensitivity.csv"):responses.append(f)
            for f in loadcsv(folder/"per_class_and_scale.csv"):classes.append(f)
            for f in loadcsv(folder/"geometry_distributions.csv"):geometry.append(f)
            for x in pred:perimage.append(x)
            assert len(loadcsv(folder/"training_time.csv"))==4
            assert len(loadcsv(folder/"scale_sensitivity.csv"))==4*2*5*3
    assert len(allsummary)==len(expected)*3
    assert len(expected)==6
    writecsv(out/"per_seed_overall.csv",allsummary)
    writecsv(out/"per_fold_group_metrics.csv",fold)
    writecsv(out/"pairwise_confusion_directions.csv",pairs)
    writecsv(out/"all_class_error_rates_raw.csv",class_error_rows)
    writecsv(out/"family_error_rates_raw.csv",family_error_rows)
    writecsv(out/"five_scale_group_responses.csv",responses)
    writecsv(out/"five_scale_16class_responses.csv",classes)
    writecsv(out/"trained_geometry.csv",geometry)
    writecsv(out/"oof_raw_predictions.csv",perimage)
    stats=[]
    for arm in expected:
        v=[x for x in allsummary if x["arm"]==arm]
        total=np.mean([x["accuracy"] for x in v])
        val=[x for x in fold if x["arm"]==arm and x["dataset"]=="OfficialVal"
            and x["group"]=="all"]
        normal=[x for x in responses if x["arm"]==arm and x["dataset"]=="OOF"
            and x["group"] in ("digits","letters")]
        risk=[]
        for group,scale in (("digits",1.2),("letters",.8)):
            current=[float(x["family_flip"]) for x in normal
                     if x["group"]==group and abs(float(x["scale"])-scale)<1e-6]
            assert len(current)==len(SEEDS)*len(WRITERS)
            risk.append(float(np.mean(current)))
        fcs=[x for x in classes if x["arm"]==arm and x["dataset"]=="OOF"
             and x["actual"] in ("B","8","D") and float(x["scale"]) in (.8,1.2)]
        worst=max(float(x["family_flip"]) for x in fcs) # single writer/seed, disclose
        fco=[x for x in classes if x["arm"]==arm and x["dataset"]=="OfficialVal"
             and x["actual"] in ("B","8","D") and float(x["scale"]) in (.8,1.2)]
        ov_worst=max(float(x["family_flip"]) for x in fco)
        stats.append({"arm":arm,"size_ratio":ARMS[arm][0],
            "stroke_strength":ARMS[arm][1],
            "mean_accuracy":total,
            "std_accuracy_seed":float(np.std([x["accuracy"] for x in v])),
            "mean_digits_acc":float(np.mean([x["digits_acc"] for x in v])),
            "mean_letters_acc":float(np.mean([x["letters_acc"] for x in v])),
            "official_val_accuracy":float(np.mean([float(x["accuracy"]) for x in val])),
            "digits_1p2_family_flip":risk[0],
            "letters_0p8_family_flip":risk[1],
            "max_writer_seed_B8D_family_flip":worst,
            "max_official_fold_seed_B8D_family_flip":ov_worst})
    writecsv(out/"comparison.csv",stats)
    # All 240 directional errors: aggregate across 4 held-out writers,
    # then average across 3 seeds. Percentages always use the source class
    # denominator (160 Hand-made per class per seed).
    baselinepath=Path(__file__).resolve().parents[4]/"05_model_optimize"/"MLP"/"Combined"/"results"/"EXP_CMB_001"/"OOF_predictions.csv"
    baseline=loadcsv(baselinepath)
    assert len(baseline)==2560
    baseline_cm=np.zeros((16,16),dtype=int)
    for r in baseline:
        aa=LABELS.index(r["true_label"]);bb=LABELS.index(r["combined_pred"])
        baseline_cm[aa,bb]+=1
    assert (np.trace(baseline_cm),baseline_cm.sum())==(2080,2560)
    direction_comparisons=[]
    class_comparisons=[]
    for arm in expected:
        baseline_wrong={LABELS[i]:int(baseline_cm[i].sum()-baseline_cm[i,i]) for i in range(16)}
        arr=[r for r in pairs if r["arm"]==arm and r["dataset"]=="OOF_original"]
        rr=[r for r in class_error_rows if r["arm"]==arm and r["dataset"]=="OOF_original"]
        assert len(arr)==3*4*16*15,(arm,len(arr))
        assert len(rr)==3*4*16
        for i,letter in enumerate(LABELS):
            cr=[r for r in rr if r["actual"]==letter]
            assert len(cr)==12
            current=float(np.mean([float(r["class_error_rate"]) for r in cr]))
            base=float(baseline_wrong[letter]/160)
            class_comparisons.append({"arm":arm,"actual":letter,"n_actual_per_seed":160,
               "baseline_wrong":baseline_wrong[letter],
               "baseline_error_rate":base,"mean_wrong":float(np.mean(
                    [sum(int(z["wrong"]) for z in cr if int(z["seed"])==seed) for seed in SEEDS])),
               "mean_class_error_rate":current,"delta_error_rate_pp":100*(current-base)})
            for j,target in enumerate(LABELS):
                if i==j:continue
                group=[r for r in arr if r["actual"]==letter and r["predicted"]==target]
                assert len(group)==12,(arm,letter,target,len(group))
                seedtot=[sum(int(r["errors"]) for r in group if int(r["seed"])==seed) for seed in SEEDS]
                baseline_count=int(baseline_cm[i,j])
                avg=float(np.mean(seedtot))
                direction_comparisons.append({"arm":arm,"actual":letter,"predicted":target,
                    "n_actual_per_seed":160,"baseline_errors":baseline_count,
                    "baseline_rate":baseline_count/160,
                    "mean_errors":avg,"mean_error_rate":avg/160,
                    "delta_errors_vs_baseline":avg-baseline_count,
                    "delta_error_rate_pp":100*(avg-baseline_count)/160})
    writecsv(out/"all_240_direction_changes.csv",direction_comparisons)
    writecsv(out/"all_16_class_error_changes.csv",class_comparisons)
    # Every direction, even absent, remains in the CSV; top20 for quick review.
    top20=sorted(direction_comparisons,key=lambda r:(-r["mean_errors"],r["arm"],r["actual"],r["predicted"]))[:20]
    writecsv(out/"top_20_error_directions_across_conditions.csv",top20)
    family_summary=[]
    for arm in expected:
        for fam in ("digit_to_letter","letter_to_digit","digit_to_digit","letter_to_letter"):
            x=[r for r in family_error_rows if r["arm"]==arm and r["dataset"]=="OOF_original"
               and r["direction"]==fam]
            assert len(x)==12
            family_summary.append({"arm":arm,"direction":fam,
              "mean_error_count":float(np.mean([sum(int(z["errors"]) for z in x
                                            if int(z["seed"])==seed) for seed in SEEDS])),
              "mean_error_rate_by_actual_family":float(np.mean([float(z["error_rate"]) for z in x]))})
    writecsv(out/"family_error_direction_comparison.csv",family_summary)

    # Paired effects: retain same seed/writer/image index for all 40/50 conditions.
    corrections=[]
    for i in range(len(expected)):
        for j in range(i+1,len(expected)):
            a=expected[i];b=expected[j]
            for seed in SEEDS:
                def get(am):
                    return {(int(x["writer"]),int(x["index"])):x
                         for x in perimage if x["arm"]==am and int(x["seed"])==seed}
                pa,pb=get(a),get(b)
                assert len(pa)==len(pb)==2560 and pa.keys()==pb.keys()
                old=np.array([pa[k]["pred"]==pa[k]["actual"] for k in sorted(pa)])
                new=np.array([pb[k]["pred"]==pb[k]["actual"] for k in sorted(pb)])
                corrections.append({"a":a,"b":b,"seed":seed,
                    "corrected":int(np.sum(~old&new)),
                    "induced":int(np.sum(old&~new)),
                    "net":int(new.sum()-old.sum())})
    writecsv(out/"paired_corrected_induced.csv",corrections)
    lines=[f"# EXP-CMB-006 — {args.stage} 실측 집계","",
      "**전처리 효과 분석용. MLP HP는 모든 조건에 고정. 특정 Accuracy/F1 숫자를 넘기면 합격시키는 연구가 아니다.**",
      "", "| 조건 | OOF Accuracy | 숫자 | 문자 | 공식 Val | 숫자×1.2 Family flip | 문자×0.8 Family flip |",
      "|---|---:|---:|---:|---:|---:|---:|"]
    for a in stats:
        lines.append("| {} | {:.3%} | {:.3%} | {:.3%} | {:.3%} | {:.2%} | {:.2%} |".format(
          a["arm"],a["mean_accuracy"],a["mean_digits_acc"],a["mean_letters_acc"],
          a["official_val_accuracy"],a["digits_1p2_family_flip"],
          a["letters_0p8_family_flip"]))
    lines+=["","## 16개 전체 클래스의 오분류율 (OOF, 각 실제 클래스 160장/Seed)",
      "","| 조건 | 실제 클래스 | Baseline 오분류율 | 새 모델 오분류율 | 변화 (pp) |",
      "|---|---|---:|---:|---:|"]
    for r in class_comparisons:
        lines.append("| {} | {} | {:.2%} | {:.2%} | {:+.2f} |".format(
            r["arm"],r["actual"],r["baseline_error_rate"],
            r["mean_class_error_rate"],r["delta_error_rate_pp"]))
    lines+=["","## 그 밖의 주요 오류 방향 (기존 Baseline에서 0↔D/8↔B 제외 상위 15개)",
      "","| 실제→예측 | 기존 건수/160 | 기존 방향 오류율 |"]
    lines+=["|---|---:|---:|"]
    other=[]
    for i,a in enumerate(LABELS):
        for j,b in enumerate(LABELS):
            if i==j or (a,b) in (("0","D"),("D","0"),("8","B"),("B","8")):
                continue
            other.append((int(baseline_cm[i,j]),a,b))
    for count,a,b in sorted(other,reverse=True)[:15]:
        lines.append("| {}→{} | {} | {:.2%} |".format(a,b,count,count/160))
    lines+=["","**모든 방향(16×15=240개)과 전처리 전후 변화는 [all_240_direction_changes.csv](all_240_direction_changes.csv).**",
      "**전체 클래스별 오류율과 0→D/8→B 외 새롭게 늘어난 혼동은 [all_16_class_error_changes.csv](all_16_class_error_changes.csv), [family_error_direction_comparison.csv](family_error_direction_comparison.csv)에도 기록.**",
      ""]
    lines+=["","**주의:** 표의 숫자/문자 Family flip은 이미지는 같은데 문자계열 P(letter)≥0.5 여부가 바뀐 비율. Accuracy와 다르다. 개별 클래스/작성자에서 반대 방향의 오류와 공식 Val 하락이 존재할 수 있다.",
      "",
      "관련 원본: per_seed_overall.csv, per_fold_group_metrics.csv, five_scale_16class_responses.csv, trained_geometry.csv, pairwise_confusion_directions.csv, paired_corrected_induced.csv.",
      "",
      "사전등록된 비교 질문에 대한 기전 해석/반례/연구 목적 달성 판정은 별도 최종 연구일지에서 한다. 원본 Test와 교수님 Test는 접근하지 않았다.", ""]
    (out/"SUMMARY.md").write_text("\n".join(lines),encoding="utf-8")
    fig,ax=plt.subplots(figsize=(10,5.2))
    xs=np.arange(len(stats))
    ax.plot(xs,[100*a["mean_accuracy"] for a in stats],marker="o",label="Held-out writer OOF")
    ax.plot(xs,[100*a["official_val_accuracy"] for a in stats],marker="s",label="Official Development Val")
    ax.set(xticks=xs,xticklabels=[a["arm"] for a in stats],
       ylabel="Accuracy (%)",title=f"EXP-CMB-006 {args.stage} | Fixed MLP HP")
    plt.setp(ax.get_xticklabels(),rotation=30,ha="right")
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(out/"comparison.png",dpi=170);plt.close(fig)
    (out/"manifest.json").write_text(json.dumps({"experiment":"EXP-CMB-006",
      "stage":args.stage,"arms":expected,"seeds":SEEDS,"folds":WRITERS,
      "expected_fits":len(expected)*len(SEEDS)*len(WRITERS),
      "no_hp_search":True,"no_test_used":True,
      "research_gate":"explanation not absolute accuracy",
      "status":"numeric comparison complete, explanatory decision pending",
      "reference_experiment":"EXP-CMB-005",
      "frozen_mlp_parameters":True,
      "original_0_to_30_not_retrained":True},
      ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)
if __name__=="__main__":
    main()


## exp_cmb006_cross_compare

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb006_cross_compare.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb006_cross_compare.py)  
Git blob: `50ad41093ac9b202baa12db668f4e16a42b6a8f9`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-006: compare frozen EXP-CMB-005 0-30% against new 40-50%.

All original OOF predictions come from previously merged main results; no
retraining, renormalization, or test-set inspection takes place here.

CLI 인자 (정적 추출): `--old`, `--new`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb006_cross_compare.py
#!/usr/bin/env python3
"""EXP-CMB-006: compare frozen EXP-CMB-005 0-30% against new 40-50%.

All original OOF predictions come from previously merged main results; no
retraining, renormalization, or test-set inspection takes place here.
"""
from __future__ import annotations
import argparse,csv,json
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

LABELS="0123456789ABCDEF"
ORIG_ARMS=["com_only"]+[f"scale{pct:02d}_stroke{stroke:02d}" for pct in (10,20,30) for stroke in (0,14,28)]
NEW_ARMS=[f"scale{pct:02d}_stroke{stroke:02d}" for pct in (40,50) for stroke in (0,14,28)]
REFERENCES=("scale30_stroke00","scale30_stroke28")
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)

def rows(path):
    with path.open(encoding="utf-8-sig",newline="") as file:
        return list(csv.DictReader(file))

def csvout(path,data):
    assert data
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(data[0]));w.writeheader();w.writerows(data)

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--old",type=Path,required=True)
    parser.add_argument("--new",type=Path,required=True)
    parser.add_argument("--out",type=Path,required=True)
    args=parser.parse_args()
    old,new,out=args.old,args.new,args.out
    out.mkdir(parents=True,exist_ok=True)

    a=rows(old/"comparison.csv");b=rows(new/"comparison.csv")
    assert [x["arm"] for x in a]==ORIG_ARMS, [x["arm"] for x in a]
    assert [x["arm"] for x in b]==NEW_ARMS, [x["arm"] for x in b]
    combined=a+b
    assert len(combined)==16 and len({x["arm"] for x in combined})==16
    csvout(out/"all_16_preprocessing_conditions.csv",combined)

    classes=rows(old/"all_16_class_error_changes.csv")+rows(new/"all_16_class_error_changes.csv")
    dirs=rows(old/"all_240_direction_changes.csv")+rows(new/"all_240_direction_changes.csv")
    assert len(classes)==16*16 and len(dirs)==16*240
    csvout(out/"all_16_class_error_rates_across_16_conditions.csv",classes)
    csvout(out/"all_240_directed_errors_across_16_conditions.csv",dirs)

    a_pred=rows(old/"oof_raw_predictions.csv")
    b_pred=rows(new/"oof_raw_predictions.csv")
    assert len(a_pred)==10*3*2560 and len(b_pred)==6*3*2560
    def organize(data):
        result=defaultdict(dict)
        for r in data:
            arm=r["arm"];seed=int(r["seed"]);writer=int(r["writer"]);idx=int(r["index"])
            k=(writer,idx);g=(arm,seed)
            assert writer in WRITERS and seed in SEEDS
            assert k not in result[g],(g,k)
            result[g][k]=(r["actual"],r["pred"])
        for key,v in result.items():
            assert len(v)==2560,(key,len(v))
        return result
    original=organize(a_pred)
    extended=organize(b_pred)
    paired=[];paired_class=[];paired_writer=[]
    for reference in REFERENCES:
        for arm in NEW_ARMS:
            for seed in SEEDS:
                prior=original[(reference,seed)];current=extended[(arm,seed)]
                assert prior.keys()==current.keys() and len(prior)==2560
                old_ok=[];new_ok=[]
                for key in sorted(prior):
                    true,old_pred=prior[key];true2,new_pred=current[key]
                    assert true==true2
                    old_ok.append(old_pred==true)
                    new_ok.append(new_pred==true)
                old_ok=np.array(old_ok,dtype=bool);new_ok=np.array(new_ok,dtype=bool)
                item={"reference":reference,"new_arm":arm,"seed":seed,
                      "n":2560,"old_correct":int(old_ok.sum()),
                      "new_correct":int(new_ok.sum()),
                      "corrected":int(np.sum(~old_ok&new_ok)),
                      "induced":int(np.sum(old_ok&~new_ok)),
                      "delta_correct":int(new_ok.sum()-old_ok.sum()),
                      "delta_accuracy_pp":float(100*(new_ok.mean()-old_ok.mean()))}
                assert item["corrected"]-item["induced"]==item["delta_correct"]
                paired.append(item)
                for actual in LABELS:
                    keys=[k for k in prior if prior[k][0]==actual]
                    assert len(keys)==160,(actual,len(keys))
                    corrected=sum(prior[k][1]!=actual and current[k][1]==actual for k in keys)
                    induced=sum(prior[k][1]==actual and current[k][1]!=actual for k in keys)
                    prior_err=sum(prior[k][1]!=actual for k in keys)
                    new_err=sum(current[k][1]!=actual for k in keys)
                    paired_class.append({"reference":reference,"new_arm":arm,"seed":seed,
                       "actual":actual,"n":160,"old_errors":prior_err,"new_errors":new_err,
                       "corrected":corrected,"induced":induced,
                       "change_error_rate_pp":float(100*(new_err-prior_err)/160)})
                for writer in WRITERS:
                    keys=[k for k in prior if k[0]==writer]
                    assert len(keys)==640
                    old_w=sum(prior[k][1]==prior[k][0] for k in keys)
                    new_w=sum(current[k][1]==current[k][0] for k in keys)
                    paired_writer.append({"reference":reference,"new_arm":arm,"seed":seed,
                       "writer":writer,"n":640,"old_correct":old_w,"new_correct":new_w,
                       "delta_accuracy_pp":float(100*(new_w-old_w)/640)})
    assert len(paired)==len(REFERENCES)*len(NEW_ARMS)*3
    csvout(out/"paired_30percent_to_40_50_corrections.csv",paired)
    csvout(out/"paired_30percent_to_40_50_per_class.csv",paired_class)
    csvout(out/"paired_30percent_to_40_50_per_writer.csv",paired_writer)

    # Compare *mean* directed counts per 2560 held-out OOF across 3 seeds,
    # keeping nonzero and zero directions as separate, explicit rows.
    directed_delta=[]
    dmap={(x["arm"],x["actual"],x["predicted"]):float(x["mean_errors"]) for x in dirs}
    for reference in REFERENCES:
        for arm in NEW_ARMS:
            for aclass in LABELS:
                for predicted in LABELS:
                    if aclass==predicted:continue
                    older=dmap[(reference,aclass,predicted)]
                    newer=dmap[(arm,aclass,predicted)]
                    directed_delta.append({"reference":reference,"new_arm":arm,
                      "actual":aclass,"predicted":predicted,
                      "old_mean_errors":older,"new_mean_errors":newer,
                      "delta_errors":newer-older,
                      "delta_actual_class_rate_pp":float(100*(newer-older)/160)})
    csvout(out/"all_240_pair_changes_vs_30percent.csv",directed_delta)
    assert len(directed_delta)==2*6*240

    # Include Normal-style family flip and official development Validation on
    # the same chart as OOF. These are *observed*, not selection thresholds.
    fig,ax=plt.subplots(figsize=(14,6))
    xs=np.arange(len(combined))
    ax.plot(xs,[100*float(r["mean_accuracy"]) for r in combined],
            marker="o",label="Held-out writer OOF")
    ax.plot(xs,[100*float(r["official_val_accuracy"]) for r in combined],
            marker="s",label="Official development Validation")
    ax.set(xticks=xs,xticklabels=[r["arm"] for r in combined],
           ylabel="Accuracy (%)",title="EXP-CMB-005+006 | frozen MLP | Size 0–50%")
    plt.setp(ax.get_xticklabels(),rotation=42,ha="right")
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(out/"size0to50_oof_vs_official.png",dpi=160)
    plt.close(fig)

    notes=["# Combined Size 0–50% × Stroke 0/14/28 — 16조건 실측 비교",
      "",
      "**모든 모델 MLP HP 동일. EXP-CMB-005의 10조건 원본 실측 재사용, EXP-CMB-006의 6조건만 새로 학습.**",
      "연구 판정은 숫자 Accuracy/F1 하한이 아니라 원인·상충·반례 설명이 가능한지로 결정.",
      "",
      "| 조건 | OOF | 숫자 | 문자 | OfficialVal | 숫자×1.2 계열 Flip | 문자×0.8 계열 Flip |",
      "|---|---:|---:|---:|---:|---:|---:|"]
    for r in combined:
        notes.append("| {} | {:.3%} | {:.3%} | {:.3%} | {:.3%} | {:.2%} | {:.2%} |".format(
            r["arm"],float(r["mean_accuracy"]),float(r["mean_digits_acc"]),
            float(r["mean_letters_acc"]),float(r["official_val_accuracy"]),
            float(r["digits_1p2_family_flip"]),float(r["letters_0p8_family_flip"])))
    notes+=["","## 30% 기준 대비 새 오답·교정 진단 (동일 Seed × Writer × Image 대응)"]
    for reference in REFERENCES:
        notes+=["",f"### 기준 {reference}","",
                "| 새 조건 | 3Seed 평균 교정 | 3Seed 평균 신규 오답 | 평균 OOF Δ(pp) |",
                "|---|---:|---:|---:|"]
        for arm in NEW_ARMS:
            pp=[r for r in paired if r["reference"]==reference and r["new_arm"]==arm]
            notes.append("| {} | {:.1f} | {:.1f} | {:+.3f} |".format(
                arm,np.mean([r["corrected"] for r in pp]),
                np.mean([r["induced"] for r in pp]),
                np.mean([r["delta_accuracy_pp"] for r in pp])))
    notes+=["","**모든 16클래스 및 240개 방향별 악화·개선은 함께 생성한 CSV에서 확인.**",
        "새 실험이 기존 30%보다 OOF 점수가 높더라도 OfficialVal·숫자/문자·필체별 손실과 반례가 있으면 별도 서술.",
        "",
        "원본 Test/교수님 Test/Normal 신규 480장 미사용. 반복 사용한 Hand-made 4명은 독립 신규 작성자 Test가 아님."]
    (out/"SUMMARY.md").write_text("\n".join(notes)+"\n",encoding="utf-8")
    manifest={"experiment":"EXP-CMB-006","reference":"EXP-CMB-005",
      "old_condition_count":len(ORIG_ARMS),"new_condition_count":len(NEW_ARMS),
      "total_condition_count":len(combined),
      "reference_arms":REFERENCES,"seeds":SEEDS,"writers":WRITERS,
      "expected_new_fits":72,"hyperparameters_changed":False,
      "original_test_used":False,"professor_test_used":False,
      "status":"numeric comparison complete; explanatory audit pending"}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)

if __name__=="__main__":
    main()


## exp_cmb007_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb007_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb007_aggregate.py)  
Git blob: `36027cf39af66bc7aea1bfafb50f7ce310e69704`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Aggregate all 15 EXP-CMB-007 Size 60-100 x Stroke 0/14/28 runs (no training).

Each completed run has four held-out writer model predictions, 16 class error
rates, 240 directed misclassification counts, and five-scale family responses.

CLI 인자 (정적 추출): `--incoming`, `--out`, `--stage`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb007_aggregate.py
#!/usr/bin/env python3
"""Aggregate all 15 EXP-CMB-007 Size 60-100 x Stroke 0/14/28 runs (no training).

Each completed run has four held-out writer model predictions, 16 class error
rates, 240 directed misclassification counts, and five-scale family responses.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
ARMS={f"scale{pct:02d}_stroke{strength:02d}":(pct/100,strength/100)
      for pct in (60,70,80,90,100) for strength in (0,14,28)}
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
LABELS="0123456789ABCDEF"
def loadcsv(p):
    with p.open(encoding="utf-8-sig",newline="") as f:return list(csv.DictReader(f))
def writecsv(p,rows):
    rows=list(rows);assert rows,p
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=rows[0].keys());w.writeheader();w.writerows(rows)
def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--incoming",type=Path,required=True)
    parser.add_argument("--out",type=Path,required=True)
    parser.add_argument("--stage",choices=("extended",),required=True)
    args=parser.parse_args()
    expected=list(ARMS)
    out=args.out;out.mkdir(parents=True,exist_ok=True)
    original=out/"raw"
    allsummary=[];fold=[];pairs=[];class_error_rows=[];family_error_rows=[];responses=[];classes=[];geometry=[];perimage=[]
    for arm in expected:
        for seed in SEEDS:
            folder=args.incoming/f"cmb007-{arm}-s{seed}"
            manifest=json.loads((folder/"manifest.json").read_text(encoding="utf-8"))
            assert manifest["arm"]==arm and int(manifest["seed"])==seed
            assert manifest["experiment"]=="EXP-CMB-007" and manifest["fixed_model_hyperparameters"]
            assert not manifest["original_test_used"] and not manifest["professor_test_used"]
            assert not manifest["new_handmade_480_used"]
            assert manifest["no_mlp_hyperparameter_search"] if "no_mlp_hyperparameter_search" in manifest else True
            target=original/arm/f"seed_{seed}"
            target.mkdir(parents=True,exist_ok=True)
            for item in folder.iterdir():
                if item.is_file():
                    shutil.copyfile(item,target/item.name)
            pred=loadcsv(folder/"oof_predictions.csv")
            assert len(pred)==2560
            pp={int(x["writer"]):[] for x in pred}
            for x in pred:
                pp[int(x["writer"])].append(x)
            assert all(len(v)==640 for v in pp.values())
            yy=np.array([LABELS.index(x["actual"]) for x in pred])
            predicted=np.array([LABELS.index(x["pred"]) for x in pred])
            pr,re,f1,_=precision_recall_fscore_support(yy,predicted,
                labels=np.arange(16),zero_division=0)
            summary={"arm":arm,"seed":seed,
                "ratio":ARMS[arm][0],"stroke":ARMS[arm][1],
                "n":len(yy),"correct":int(np.sum(yy==predicted)),
                "accuracy":float(np.mean(yy==predicted)),
                "macro_f1":float(f1.mean()),
                "digits_acc":float(np.mean(yy[yy<10]==predicted[yy<10])),
                "letters_acc":float(np.mean(yy[yy>=10]==predicted[yy>=10]))}
            allsummary.append(summary)
            for f in loadcsv(folder/"fold_metrics.csv"):fold.append(f)
            for f in loadcsv(folder/"pair_confusions.csv"):pairs.append(f)
            for f in loadcsv(folder/"per_class_error_rates.csv"):class_error_rows.append(f)
            for f in loadcsv(folder/"family_confusion_rates.csv"):family_error_rows.append(f)
            for f in loadcsv(folder/"scale_sensitivity.csv"):responses.append(f)
            for f in loadcsv(folder/"per_class_and_scale.csv"):classes.append(f)
            for f in loadcsv(folder/"geometry_distributions.csv"):geometry.append(f)
            for x in pred:perimage.append(x)
            assert len(loadcsv(folder/"training_time.csv"))==4
            assert len(loadcsv(folder/"scale_sensitivity.csv"))==4*2*5*3
    assert len(allsummary)==len(expected)*3
    assert len(expected)==15
    writecsv(out/"per_seed_overall.csv",allsummary)
    writecsv(out/"per_fold_group_metrics.csv",fold)
    writecsv(out/"pairwise_confusion_directions.csv",pairs)
    writecsv(out/"all_class_error_rates_raw.csv",class_error_rows)
    writecsv(out/"family_error_rates_raw.csv",family_error_rows)
    writecsv(out/"five_scale_group_responses.csv",responses)
    writecsv(out/"five_scale_16class_responses.csv",classes)
    writecsv(out/"trained_geometry.csv",geometry)
    writecsv(out/"oof_raw_predictions.csv",perimage)
    stats=[]
    for arm in expected:
        v=[x for x in allsummary if x["arm"]==arm]
        total=np.mean([x["accuracy"] for x in v])
        val=[x for x in fold if x["arm"]==arm and x["dataset"]=="OfficialVal"
            and x["group"]=="all"]
        normal=[x for x in responses if x["arm"]==arm and x["dataset"]=="OOF"
            and x["group"] in ("digits","letters")]
        risk=[]
        for group,scale in (("digits",1.2),("letters",.8)):
            current=[float(x["family_flip"]) for x in normal
                     if x["group"]==group and abs(float(x["scale"])-scale)<1e-6]
            assert len(current)==len(SEEDS)*len(WRITERS)
            risk.append(float(np.mean(current)))
        fcs=[x for x in classes if x["arm"]==arm and x["dataset"]=="OOF"
             and x["actual"] in ("B","8","D") and float(x["scale"]) in (.8,1.2)]
        worst=max(float(x["family_flip"]) for x in fcs) # single writer/seed, disclose
        fco=[x for x in classes if x["arm"]==arm and x["dataset"]=="OfficialVal"
             and x["actual"] in ("B","8","D") and float(x["scale"]) in (.8,1.2)]
        ov_worst=max(float(x["family_flip"]) for x in fco)
        stats.append({"arm":arm,"size_ratio":ARMS[arm][0],
            "stroke_strength":ARMS[arm][1],
            "mean_accuracy":total,
            "std_accuracy_seed":float(np.std([x["accuracy"] for x in v])),
            "mean_digits_acc":float(np.mean([x["digits_acc"] for x in v])),
            "mean_letters_acc":float(np.mean([x["letters_acc"] for x in v])),
            "official_val_accuracy":float(np.mean([float(x["accuracy"]) for x in val])),
            "digits_1p2_family_flip":risk[0],
            "letters_0p8_family_flip":risk[1],
            "max_writer_seed_B8D_family_flip":worst,
            "max_official_fold_seed_B8D_family_flip":ov_worst})
    writecsv(out/"comparison.csv",stats)
    # All 240 directional errors: aggregate across 4 held-out writers,
    # then average across 3 seeds. Percentages always use the source class
    # denominator (160 Hand-made per class per seed).
    baselinepath=Path(__file__).resolve().parents[4]/"05_model_optimize"/"MLP"/"Combined"/"results"/"EXP_CMB_001"/"OOF_predictions.csv"
    baseline=loadcsv(baselinepath)
    assert len(baseline)==2560
    baseline_cm=np.zeros((16,16),dtype=int)
    for r in baseline:
        aa=LABELS.index(r["true_label"]);bb=LABELS.index(r["combined_pred"])
        baseline_cm[aa,bb]+=1
    assert (np.trace(baseline_cm),baseline_cm.sum())==(2080,2560)
    direction_comparisons=[]
    class_comparisons=[]
    for arm in expected:
        baseline_wrong={LABELS[i]:int(baseline_cm[i].sum()-baseline_cm[i,i]) for i in range(16)}
        arr=[r for r in pairs if r["arm"]==arm and r["dataset"]=="OOF_original"]
        rr=[r for r in class_error_rows if r["arm"]==arm and r["dataset"]=="OOF_original"]
        assert len(arr)==3*4*16*15,(arm,len(arr))
        assert len(rr)==3*4*16
        for i,letter in enumerate(LABELS):
            cr=[r for r in rr if r["actual"]==letter]
            assert len(cr)==12
            current=float(np.mean([float(r["class_error_rate"]) for r in cr]))
            base=float(baseline_wrong[letter]/160)
            class_comparisons.append({"arm":arm,"actual":letter,"n_actual_per_seed":160,
               "baseline_wrong":baseline_wrong[letter],
               "baseline_error_rate":base,"mean_wrong":float(np.mean(
                    [sum(int(z["wrong"]) for z in cr if int(z["seed"])==seed) for seed in SEEDS])),
               "mean_class_error_rate":current,"delta_error_rate_pp":100*(current-base)})
            for j,target in enumerate(LABELS):
                if i==j:continue
                group=[r for r in arr if r["actual"]==letter and r["predicted"]==target]
                assert len(group)==12,(arm,letter,target,len(group))
                seedtot=[sum(int(r["errors"]) for r in group if int(r["seed"])==seed) for seed in SEEDS]
                baseline_count=int(baseline_cm[i,j])
                avg=float(np.mean(seedtot))
                direction_comparisons.append({"arm":arm,"actual":letter,"predicted":target,
                    "n_actual_per_seed":160,"baseline_errors":baseline_count,
                    "baseline_rate":baseline_count/160,
                    "mean_errors":avg,"mean_error_rate":avg/160,
                    "delta_errors_vs_baseline":avg-baseline_count,
                    "delta_error_rate_pp":100*(avg-baseline_count)/160})
    writecsv(out/"all_240_direction_changes.csv",direction_comparisons)
    writecsv(out/"all_16_class_error_changes.csv",class_comparisons)
    # Every direction, even absent, remains in the CSV; top20 for quick review.
    top20=sorted(direction_comparisons,key=lambda r:(-r["mean_errors"],r["arm"],r["actual"],r["predicted"]))[:20]
    writecsv(out/"top_20_error_directions_across_conditions.csv",top20)
    family_summary=[]
    for arm in expected:
        for fam in ("digit_to_letter","letter_to_digit","digit_to_digit","letter_to_letter"):
            x=[r for r in family_error_rows if r["arm"]==arm and r["dataset"]=="OOF_original"
               and r["direction"]==fam]
            assert len(x)==12
            family_summary.append({"arm":arm,"direction":fam,
              "mean_error_count":float(np.mean([sum(int(z["errors"]) for z in x
                                            if int(z["seed"])==seed) for seed in SEEDS])),
              "mean_error_rate_by_actual_family":float(np.mean([float(z["error_rate"]) for z in x]))})
    writecsv(out/"family_error_direction_comparison.csv",family_summary)

    # Paired effects: retain same seed/writer/image index for all 40/50 conditions.
    corrections=[]
    for i in range(len(expected)):
        for j in range(i+1,len(expected)):
            a=expected[i];b=expected[j]
            for seed in SEEDS:
                def get(am):
                    return {(int(x["writer"]),int(x["index"])):x
                         for x in perimage if x["arm"]==am and int(x["seed"])==seed}
                pa,pb=get(a),get(b)
                assert len(pa)==len(pb)==2560 and pa.keys()==pb.keys()
                old=np.array([pa[k]["pred"]==pa[k]["actual"] for k in sorted(pa)])
                new=np.array([pb[k]["pred"]==pb[k]["actual"] for k in sorted(pb)])
                corrections.append({"a":a,"b":b,"seed":seed,
                    "corrected":int(np.sum(~old&new)),
                    "induced":int(np.sum(old&~new)),
                    "net":int(new.sum()-old.sum())})
    writecsv(out/"paired_corrected_induced.csv",corrections)
    lines=[f"# EXP-CMB-007 — {args.stage} 실측 집계","",
      "**전처리 효과 분석용. MLP HP는 모든 조건에 고정. 특정 Accuracy/F1 숫자를 넘기면 합격시키는 연구가 아니다.**",
      "", "| 조건 | OOF Accuracy | 숫자 | 문자 | 공식 Val | 숫자×1.2 Family flip | 문자×0.8 Family flip |",
      "|---|---:|---:|---:|---:|---:|---:|"]
    for a in stats:
        lines.append("| {} | {:.3%} | {:.3%} | {:.3%} | {:.3%} | {:.2%} | {:.2%} |".format(
          a["arm"],a["mean_accuracy"],a["mean_digits_acc"],a["mean_letters_acc"],
          a["official_val_accuracy"],a["digits_1p2_family_flip"],
          a["letters_0p8_family_flip"]))
    lines+=["","## 16개 전체 클래스의 오분류율 (OOF, 각 실제 클래스 160장/Seed)",
      "","| 조건 | 실제 클래스 | Baseline 오분류율 | 새 모델 오분류율 | 변화 (pp) |",
      "|---|---|---:|---:|---:|"]
    for r in class_comparisons:
        lines.append("| {} | {} | {:.2%} | {:.2%} | {:+.2f} |".format(
            r["arm"],r["actual"],r["baseline_error_rate"],
            r["mean_class_error_rate"],r["delta_error_rate_pp"]))
    lines+=["","## 그 밖의 주요 오류 방향 (기존 Baseline에서 0↔D/8↔B 제외 상위 15개)",
      "","| 실제→예측 | 기존 건수/160 | 기존 방향 오류율 |"]
    lines+=["|---|---:|---:|"]
    other=[]
    for i,a in enumerate(LABELS):
        for j,b in enumerate(LABELS):
            if i==j or (a,b) in (("0","D"),("D","0"),("8","B"),("B","8")):
                continue
            other.append((int(baseline_cm[i,j]),a,b))
    for count,a,b in sorted(other,reverse=True)[:15]:
        lines.append("| {}→{} | {} | {:.2%} |".format(a,b,count,count/160))
    lines+=["","**모든 방향(16×15=240개)과 전처리 전후 변화는 [all_240_direction_changes.csv](all_240_direction_changes.csv).**",
      "**전체 클래스별 오류율과 0→D/8→B 외 새롭게 늘어난 혼동은 [all_16_class_error_changes.csv](all_16_class_error_changes.csv), [family_error_direction_comparison.csv](family_error_direction_comparison.csv)에도 기록.**",
      ""]
    lines+=["","**주의:** 표의 숫자/문자 Family flip은 이미지는 같은데 문자계열 P(letter)≥0.5 여부가 바뀐 비율. Accuracy와 다르다. 개별 클래스/작성자에서 반대 방향의 오류와 공식 Val 하락이 존재할 수 있다.",
      "",
      "관련 원본: per_seed_overall.csv, per_fold_group_metrics.csv, five_scale_16class_responses.csv, trained_geometry.csv, pairwise_confusion_directions.csv, paired_corrected_induced.csv.",
      "",
      "사전등록된 비교 질문에 대한 기전 해석/반례/연구 목적 달성 판정은 별도 최종 연구일지에서 한다. 원본 Test와 교수님 Test는 접근하지 않았다.", ""]
    (out/"SUMMARY.md").write_text("\n".join(lines),encoding="utf-8")
    fig,ax=plt.subplots(figsize=(10,5.2))
    xs=np.arange(len(stats))
    ax.plot(xs,[100*a["mean_accuracy"] for a in stats],marker="o",label="Held-out writer OOF")
    ax.plot(xs,[100*a["official_val_accuracy"] for a in stats],marker="s",label="Official Development Val")
    ax.set(xticks=xs,xticklabels=[a["arm"] for a in stats],
       ylabel="Accuracy (%)",title=f"EXP-CMB-007 {args.stage} | Frozen Combined MLP HP")
    plt.setp(ax.get_xticklabels(),rotation=30,ha="right")
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(out/"comparison.png",dpi=170);plt.close(fig)
    (out/"manifest.json").write_text(json.dumps({"experiment":"EXP-CMB-007",
      "stage":args.stage,"arms":expected,"seeds":SEEDS,"folds":WRITERS,
      "expected_fits":len(expected)*len(SEEDS)*len(WRITERS),
      "no_hp_search":True,"no_test_used":True,
      "research_gate":"explanation not absolute accuracy",
      "status":"numeric comparison complete, explanatory decision pending",
      "reference_experiment":"EXP-CMB-005/006",
      "frozen_mlp_parameters":True,
      "original_0_to_50_not_retrained":True},
      ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)
if __name__=="__main__":
    main()


## exp_cmb007_cross_compare

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb007_cross_compare.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb007_cross_compare.py)  
Git blob: `7807333683e199a4194fcfab59588f4ab2cfc43f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-007: compare frozen EXP-CMB-005/006 frozen 0-50% against EXP-CMB-007 new 60-100%.

All original OOF predictions come from previously merged main results; no
retraining, renormalization, or test-set inspection takes place here.

CLI 인자 (정적 추출): `--old`, `--old-low`, `--old-high`, `--new`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb007_cross_compare.py
#!/usr/bin/env python3
"""EXP-CMB-007: compare frozen EXP-CMB-005/006 frozen 0-50% against EXP-CMB-007 new 60-100%.

All original OOF predictions come from previously merged main results; no
retraining, renormalization, or test-set inspection takes place here.
"""
from __future__ import annotations
import argparse,csv,json
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

LABELS="0123456789ABCDEF"
ORIG_ARMS=["com_only"]+[f"scale{pct:02d}_stroke{stroke:02d}" for pct in (10,20,30,40,50) for stroke in (0,14,28)]
NEW_ARMS=[f"scale{pct:02d}_stroke{stroke:02d}" for pct in (60,70,80,90,100) for stroke in (0,14,28)]
REFERENCES=("scale30_stroke00","scale30_stroke28","scale50_stroke00","scale50_stroke28")
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)

def rows(path):
    with path.open(encoding="utf-8-sig",newline="") as file:
        return list(csv.DictReader(file))

def csvout(path,data):
    assert data
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(data[0]));w.writeheader();w.writerows(data)

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--old",type=Path,required=True)
    parser.add_argument("--old-low",type=Path,required=True)
    parser.add_argument("--old-high",type=Path,required=True)
    parser.add_argument("--new",type=Path,required=True)
    parser.add_argument("--out",type=Path,required=True)
    args=parser.parse_args()
    old,new,out=args.old,args.new,args.out
    out.mkdir(parents=True,exist_ok=True)

    a=rows(old/"all_16_preprocessing_conditions.csv");b=rows(new/"comparison.csv")
    assert [x["arm"] for x in a]==ORIG_ARMS, [x["arm"] for x in a]
    assert [x["arm"] for x in b]==NEW_ARMS, [x["arm"] for x in b]
    combined=a+b
    assert len(combined)==31 and len({x["arm"] for x in combined})==31
    csvout(out/"all_31_preprocessing_conditions.csv",combined)

    classes=rows(old/"all_16_class_error_rates_across_16_conditions.csv")+rows(new/"all_16_class_error_changes.csv")
    dirs=rows(old/"all_240_directed_errors_across_16_conditions.csv")+rows(new/"all_240_direction_changes.csv")
    assert len(classes)==31*16 and len(dirs)==31*240
    csvout(out/"all_16_class_error_rates_across_31_conditions.csv",classes)
    csvout(out/"all_240_directed_errors_across_31_conditions.csv",dirs)

    a_pred=rows(args.old_low/"oof_raw_predictions.csv")+rows(args.old_high/"oof_raw_predictions.csv")
    b_pred=rows(new/"oof_raw_predictions.csv")
    assert len(a_pred)==16*3*2560 and len(b_pred)==15*3*2560
    def organize(data):
        result=defaultdict(dict)
        for r in data:
            arm=r["arm"];seed=int(r["seed"]);writer=int(r["writer"]);idx=int(r["index"])
            k=(writer,idx);g=(arm,seed)
            assert writer in WRITERS and seed in SEEDS
            assert k not in result[g],(g,k)
            result[g][k]=(r["actual"],r["pred"])
        for key,v in result.items():
            assert len(v)==2560,(key,len(v))
        return result
    original=organize(a_pred)
    extended=organize(b_pred)
    paired=[];paired_class=[];paired_writer=[]
    for reference in REFERENCES:
        for arm in NEW_ARMS:
            for seed in SEEDS:
                prior=original[(reference,seed)];current=extended[(arm,seed)]
                assert prior.keys()==current.keys() and len(prior)==2560
                old_ok=[];new_ok=[]
                for key in sorted(prior):
                    true,old_pred=prior[key];true2,new_pred=current[key]
                    assert true==true2
                    old_ok.append(old_pred==true)
                    new_ok.append(new_pred==true)
                old_ok=np.array(old_ok,dtype=bool);new_ok=np.array(new_ok,dtype=bool)
                item={"reference":reference,"new_arm":arm,"seed":seed,
                      "n":2560,"old_correct":int(old_ok.sum()),
                      "new_correct":int(new_ok.sum()),
                      "corrected":int(np.sum(~old_ok&new_ok)),
                      "induced":int(np.sum(old_ok&~new_ok)),
                      "delta_correct":int(new_ok.sum()-old_ok.sum()),
                      "delta_accuracy_pp":float(100*(new_ok.mean()-old_ok.mean()))}
                assert item["corrected"]-item["induced"]==item["delta_correct"]
                paired.append(item)
                for actual in LABELS:
                    keys=[k for k in prior if prior[k][0]==actual]
                    assert len(keys)==160,(actual,len(keys))
                    corrected=sum(prior[k][1]!=actual and current[k][1]==actual for k in keys)
                    induced=sum(prior[k][1]==actual and current[k][1]!=actual for k in keys)
                    prior_err=sum(prior[k][1]!=actual for k in keys)
                    new_err=sum(current[k][1]!=actual for k in keys)
                    paired_class.append({"reference":reference,"new_arm":arm,"seed":seed,
                       "actual":actual,"n":160,"old_errors":prior_err,"new_errors":new_err,
                       "corrected":corrected,"induced":induced,
                       "change_error_rate_pp":float(100*(new_err-prior_err)/160)})
                for writer in WRITERS:
                    keys=[k for k in prior if k[0]==writer]
                    assert len(keys)==640
                    old_w=sum(prior[k][1]==prior[k][0] for k in keys)
                    new_w=sum(current[k][1]==current[k][0] for k in keys)
                    paired_writer.append({"reference":reference,"new_arm":arm,"seed":seed,
                       "writer":writer,"n":640,"old_correct":old_w,"new_correct":new_w,
                       "delta_accuracy_pp":float(100*(new_w-old_w)/640)})
    assert len(paired)==len(REFERENCES)*len(NEW_ARMS)*3
    csvout(out/"paired_30_50_vs_60_100_corrections.csv",paired)
    csvout(out/"paired_30_50_vs_60_100_per_class.csv",paired_class)
    csvout(out/"paired_30_50_vs_60_100_per_writer.csv",paired_writer)

    # Compare *mean* directed counts per 2560 held-out OOF across 3 seeds,
    # keeping nonzero and zero directions as separate, explicit rows.
    directed_delta=[]
    dmap={(x["arm"],x["actual"],x["predicted"]):float(x["mean_errors"]) for x in dirs}
    for reference in REFERENCES:
        for arm in NEW_ARMS:
            for aclass in LABELS:
                for predicted in LABELS:
                    if aclass==predicted:continue
                    older=dmap[(reference,aclass,predicted)]
                    newer=dmap[(arm,aclass,predicted)]
                    directed_delta.append({"reference":reference,"new_arm":arm,
                      "actual":aclass,"predicted":predicted,
                      "old_mean_errors":older,"new_mean_errors":newer,
                      "delta_errors":newer-older,
                      "delta_actual_class_rate_pp":float(100*(newer-older)/160)})
    csvout(out/"all_240_pair_changes_vs_30_50.csv",directed_delta)
    assert len(directed_delta)==len(REFERENCES)*len(NEW_ARMS)*240


    # Family-flip stability covers all 16 classes, both data sources, 0.8-1.2.
    # 1.0 is the frozen model's per-image reference. Keep source class and
    # writer-by-seed worst-case separate from class-mean flip.
    raw_class=(rows(args.old_low/"five_scale_16class_responses.csv")
             +rows(args.old_high/"five_scale_16class_responses.csv")
             +rows(new/"five_scale_16class_responses.csv"))
    expected_raw=31*3*4*2*5*16
    assert len(raw_class)==expected_raw,(len(raw_class),expected_raw)
    class_group=defaultdict(list)
    for r in raw_class:
        arm,source,actual,scale=r["arm"],r["dataset"],r["actual"],float(r["scale"])
        assert arm in (ORIG_ARMS+NEW_ARMS) and source in ("OOF","OfficialVal")
        assert actual in LABELS
        class_group[(arm,source,actual,scale)].append(float(r["family_flip"]))
    class_flip=[]
    for (arm,source,actual,scale),v in class_group.items():
        assert len(v)==3*4,(arm,source,actual,scale,len(v))
        class_flip.append({"arm":arm,"dataset":source,"actual":actual,"scale":scale,
                           "mean_family_flip":float(np.mean(v)),
                           "max_fold_seed_family_flip":float(max(v)),
                           "min_fold_seed_family_flip":float(min(v))})
    assert len(class_flip)==31*2*16*5
    class_flip.sort(key=lambda x:((ORIG_ARMS+NEW_ARMS).index(x["arm"]),
                              x["dataset"],x["actual"],x["scale"]))
    csvout(out/"all_31_condition_16class_five_scale_flip.csv",class_flip)
    worst=[]
    for arm in (ORIG_ARMS+NEW_ARMS):
        for source in ("OOF","OfficialVal"):
            d=[x for x in class_flip if x["arm"]==arm and x["dataset"]==source
               and x["scale"] in (.8,1.2)]
            focus=[x for x in d if x["actual"] in "B8D"]
            assert len(d)==32 and len(focus)==6
            worst.append({"arm":arm,"dataset":source,
                          "max_16class_mean_flip":float(max(x["mean_family_flip"] for x in d)),
                          "max_B8D_mean_flip":float(max(x["mean_family_flip"] for x in focus)),
                          "max_16class_fold_seed_flip":float(max(x["max_fold_seed_family_flip"] for x in d)),
                          "max_B8D_fold_seed_flip":float(max(x["max_fold_seed_family_flip"] for x in focus))})
    csvout(out/"all_31_condition_16class_worst_flip.csv",worst)

    # Include Normal-style family flip and official development Validation on
    # the same chart as OOF. These are *observed*, not selection thresholds.
    fig,ax=plt.subplots(figsize=(14,6))
    xs=np.arange(len(combined))
    ax.plot(xs,[100*float(r["mean_accuracy"]) for r in combined],
            marker="o",label="Held-out writer OOF")
    ax.plot(xs,[100*float(r["official_val_accuracy"]) for r in combined],
            marker="s",label="Official development Validation")
    ax.set(xticks=xs,xticklabels=[r["arm"] for r in combined],
           ylabel="Accuracy (%)",title="EXP-CMB-005/006+006 | frozen MLP | Size 0–100%")
    plt.setp(ax.get_xticklabels(),rotation=42,ha="right")
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(out/"size0to100_oof_vs_official.png",dpi=160)
    plt.close(fig)

    notes=["# Combined Size 0–100% × Stroke 0/14/28 — 31조건 실측 비교",
      "",
      "**모든 모델 MLP HP 동일. EXP-CMB-005/006의 16조건 원본 실측 재사용, EXP-CMB-007의 15조건만 새로 학습.**",
      "연구 판정은 숫자 Accuracy/F1 하한이 아니라 원인·상충·반례 설명이 가능한지로 결정.",
      "",
      "| 조건 | OOF | 숫자 | 문자 | OfficialVal | 숫자×1.2 계열 Flip | 문자×0.8 계열 Flip |",
      "|---|---:|---:|---:|---:|---:|---:|"]
    for r in combined:
        notes.append("| {} | {:.3%} | {:.3%} | {:.3%} | {:.3%} | {:.2%} | {:.2%} |".format(
            r["arm"],float(r["mean_accuracy"]),float(r["mean_digits_acc"]),
            float(r["mean_letters_acc"]),float(r["official_val_accuracy"]),
            float(r["digits_1p2_family_flip"]),float(r["letters_0p8_family_flip"])))
    notes+=["","## 30%/50% 기준 대비 새 오답·교정 진단 (동일 Seed × Writer × Image 대응)"]
    for reference in REFERENCES:
        notes+=["",f"### 기준 {reference}","",
                "| 새 조건 | 3Seed 평균 교정 | 3Seed 평균 신규 오답 | 평균 OOF Δ(pp) |",
                "|---|---:|---:|---:|"]
        for arm in NEW_ARMS:
            pp=[r for r in paired if r["reference"]==reference and r["new_arm"]==arm]
            notes.append("| {} | {:.1f} | {:.1f} | {:+.3f} |".format(
                arm,np.mean([r["corrected"] for r in pp]),
                np.mean([r["induced"] for r in pp]),
                np.mean([r["delta_accuracy_pp"] for r in pp])))
    notes+=["","**모든 16클래스 및 240개 오분류 방향별 악화·개선과 0.8~1.2배 개별 클래스 최악 Flip은 CSV로 확인.**",
        "새 실험이 기존 30/50%보다 OOF 점수가 높더라도 OfficialVal·숫자/문자·필체별 손실과 반례가 있으면 별도 서술.",
        "",
        "원본 Test/교수님 Test/Normal 신규 480장 미사용. 반복 사용한 Hand-made 4명은 독립 신규 작성자 Test가 아님."]
    (out/"SUMMARY.md").write_text("\n".join(notes)+"\n",encoding="utf-8")
    manifest={"experiment":"EXP-CMB-007","reference":"EXP-CMB-005/006",
      "old_condition_count":len(ORIG_ARMS),"new_condition_count":len(NEW_ARMS),
      "total_condition_count":len(combined),
      "reference_arms":REFERENCES,"seeds":SEEDS,"writers":WRITERS,
      "expected_new_fits":180,"hyperparameters_changed":False,
      "original_test_used":False,"professor_test_used":False,
      "status":"numeric comparison complete; explanatory audit pending"}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)

if __name__=="__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")